## Predicting Spotify Genre using Song Lyrics

In [2]:
import kagglehub
import os
import pandas as pd

/Users/tylerh/Downloads/Spotify-Genre-Prediction/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load data

In [3]:
data_dir = kagglehub.dataset_download("serkantysz/550k-spotify-songs-audio-lyrics-and-genres")
print("Dataset folder:", data_dir)
print("Files:", os.listdir(data_dir))

df = pd.read_csv(os.path.join(data_dir, "songs.csv"))
print("\nRaw shape:", df.shape)
print("All columns:", list(df.columns))

Dataset folder: /Users/tylerh/.cache/kagglehub/datasets/serkantysz/550k-spotify-songs-audio-lyrics-and-genres/versions/1
Files: ['songs.csv', 'artists.csv']

Raw shape: (550622, 24)
All columns: ['id', 'name', 'album_name', 'artists', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'artist_ids', 'niche_genres']


## Keep needed columns, drop rows without lyrics, save

In [4]:
KEEP_COLS = ["name", "artists", "lyrics", "genre"]
os.makedirs("data", exist_ok=True)
os.makedirs("model", exist_ok=True)
OUT_PATH = "data/songs_clean.csv"

df = df[KEEP_COLS]

before = len(df)
df = df[df["lyrics"].notna() & (df["lyrics"].astype(str).str.strip() != "")]
df = df.reset_index(drop=True)
print(f"Dropped {before - len(df):,} rows with no lyrics ({len(df):,} remain)")

df.to_csv(OUT_PATH, index=False)
print(f"Saved to {OUT_PATH}")

Dropped 0 rows with no lyrics (550,622 remain)
Saved to data/songs_clean.csv


## Quick Look at Genre Counts

In [5]:
df["genre"].value_counts()

genre
Rock          197168
Pop            72539
Electronic     69755
Folk           50843
Country        46076
Hip-Hop        41745
R&B            27337
Jazz           17010
Blues          15976
Classical      12173
Name: count, dtype: int64

### Steps Taken For Cleaning: 
replace line breaks with spaces \
Remove non-letters:, punctuation, numbers, section tags, and special characters \
Remove duplicate songs by sorting by artist then removing duplicate titles that share 90%+ lyrics \
remove stop words \
lemmatize the data

### Cleaning setup

In [6]:
import re
import numpy as np
from difflib import SequenceMatcher
from functools import lru_cache

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

df = pd.read_csv("data/songs_clean.csv")

### Steps 1-2: tags, singer labels, line breaks, non-letters

In [7]:
# Steps 1-2: tags / singer labels out, line breaks -> spaces, then strip everything that isn't a letter.
lyr = df["lyrics"]

# 0. some songs store newlines as the two characters "\n" (and quotes as \"): make them real
lyr = lyr.str.replace("\\n", "\n", regex=False).str.replace("\\r", "\n", regex=False).str.replace('\\"', '"', regex=False)

# 1. singer labels written as "NAME:" at the start of a line (PHANTOM:, Madame Giry:) -- before lowercasing,
#    because the capital letters are what tell a label from an ordinary lyric line
lyr = lyr.str.replace(r"(?m)^[ \t]*[A-Z][A-Za-z'&.-]*(?:[ \t]+[A-Z][A-Za-z'&.-]*){0,2}[ \t]*:[ \t]*", " ", regex=True)

lyr = lyr.str.lower()

# 2. bracketed tags in any casing: [Chorus], [CHORUS: Repeats], [Layzie Bone], [Incomprehensible], [?]
lyr = lyr.str.replace(r"\[[^\]]*\]", " ", regex=True)

# 3. bare section lines with no brackets: "Chorus", "chorus:", "Chorus 2x", "Instrumental", "Solo"
lyr = lyr.str.replace(
    r"(?m)^[ \t]*(?:pre-?chorus|chorus|verse|bridge|intro|outro|hook|refrain|interlude|instrumental|solo)"
    r"(?:[ \t]+(?:\d+x?|x\d+|\d+))?[ \t]*:?[ \t]*$", " ", regex=True)

# 4. line breaks -> spaces
lyr = lyr.str.replace(r"[\r\n]+", " ", regex=True)

# 5. curly apostrophes -> straight, then fold accents (cafe, senor) without breaking words
lyr = lyr.str.replace("\u2019", "'", regex=False).str.replace("\u2018", "'", regex=False)
lyr = lyr.str.normalize("NFKD").str.replace(r"[\u0300-\u036f]", "", regex=True)

# 6. everything that isn't a letter (punctuation, numbers, symbols like the music note) -> space
lyr = lyr.str.replace(r"[^a-z]+", " ", regex=True)
lyr = lyr.str.replace(r"\s+", " ", regex=True).str.strip()

df_clean = df[["name", "artists", "genre"]].copy()
df_clean["lyrics_clean"] = lyr

# songs with nothing left (instrumentals, non-Latin scripts) are useless for a lyrics model
empty = df_clean["lyrics_clean"] == ""
print(f"Dropping {empty.sum():,} songs with no usable letters")
df_clean = df_clean[~empty].reset_index(drop=True)
print(f"{len(df_clean):,} songs remain")

df_clean.to_csv("data/songs_regex_cleaned.csv", index=False)
print("Saved to data/songs_regex_cleaned.csv")

Dropping 60 songs with no usable letters
550,562 songs remain
Saved to data/songs_regex_cleaned.csv


### Step 3: remove near-duplicate songs by lyrics


Scores are computed once; tune the threshold in the cells after.

In [8]:
# Step 3: near-duplicate detection by lyrics
# Songs are sorted by artist. Within each artist, every song gets a 0-1 similarity score against the
# most similar earlier song, so versions that share 90%+ of their lyrics can be dropped.
# Scores are computed once and stored, so the threshold can be tuned in the next cells for free.
import json
from sklearn.feature_extraction.text import CountVectorizer

df_clean = pd.read_csv("data/songs_regex_cleaned.csv")   # output of steps 1-2

def primary_artist(a):
    try:
        return json.loads(a)[0]          
    except Exception:
        return a

df_dd = df_clean.copy()
df_dd["primary_artist"] = df_dd["artists"].map(primary_artist)
df_dd["title_len"] = df_dd["name"].str.len()
# shortest title first within an artist (usually the original cut), so that is the version we keep
df_dd = df_dd.sort_values(["primary_artist", "title_len"], kind="stable").reset_index(drop=True)

lyrics_col = df_dd["lyrics_clean"].to_numpy()
scores = np.zeros(len(df_dd))
partners = np.full(len(df_dd), -1)

PREFILTER = 0.5     # only run the slower exact comparison on pairs sharing >= 50% of their vocabulary
BLOCK = 1000        # rows per sparse-matrix block (keeps memory flat for artists with thousands of songs)

groups = df_dd.groupby("primary_artist", sort=False).indices
multi = [idx for idx in groups.values() if len(idx) > 1]
print(f"{len(multi):,} artists with 2+ songs to compare")

for n, idx in enumerate(multi, 1):
    texts = lyrics_col[idx]
    X = CountVectorizer(binary=True, token_pattern=r"\S+", lowercase=False).fit_transform(texts).tocsr().astype(np.float32)
    sizes = np.asarray(X.sum(axis=1)).ravel()
    XT = X.T.tocsr()
    for start in range(0, len(idx), BLOCK):
        inter = (X[start:start + BLOCK] @ XT).toarray()            # shared-vocabulary counts
        jac = inter / (sizes[start:start + BLOCK, None] + sizes[None, :] - inter)   # Jaccard of word sets
        for r in range(jac.shape[0]):
            j = start + r
            cand = np.where(jac[r, :j] >= PREFILTER)[0]             # earlier songs only
            if len(cand) == 0:
                continue
            tj = texts[j].split()
            best, best_i = 0.0, -1
            for i in cand:
                score = SequenceMatcher(None, tj, texts[i].split(), autojunk=False).ratio()
                if score > best:
                    best, best_i = score, i
            scores[idx[j]] = best
            partners[idx[j]] = idx[best_i]
    if n % 5000 == 0:
        print(f"  {n:,} / {len(multi):,} artists")

df_dd["dup_score"] = scores
df_dd["dup_partner"] = partners
print("done")

28,257 artists with 2+ songs to compare
  5,000 / 28,257 artists
  10,000 / 28,257 artists
  15,000 / 28,257 artists
  20,000 / 28,257 artists
  25,000 / 28,257 artists
done


#### Apply the threshold
This threshold is arbitrary for this notebook. It can be tuned in future study.

In [9]:
SIM_THRESHOLD = 0.90  

before = len(df_dd)
df_dd = df_dd[df_dd["dup_score"] < SIM_THRESHOLD].reset_index(drop=True)
print(f"Removed {before - len(df_dd):,} near-duplicates ({len(df_dd):,} remain)")
df_dd = df_dd[["name", "artists", "genre", "lyrics_clean"]]
df_dd.to_csv("data/songs_deduped.csv", index=False)
print("Saved to data/songs_deduped.csv")

Removed 114,789 near-duplicates (435,773 remain)
Saved to data/songs_deduped.csv


### Steps 4-5: stop words and lemmatization, then save

In [10]:
# Steps 4-5: stop words, then lemmatization.
STOP = set(stopwords.words("english"))
# Splitting "don't" on the apostrophe leaves "don" and "t", which NLTK's list also covers.
# Add lyric filler here if you want it gone too, e.g. EXTRA_STOP = {"oh", "yeah", "na", "la"}
EXTRA_STOP = set()
STOP |= EXTRA_STOP

lemmatizer = WordNetLemmatizer()

@lru_cache(maxsize=None)          # lyrics reuse a small vocabulary, so cache per word
def lemma(word):
    # no POS tagging: try verb first ("running" -> "run"), then noun ("hearts" -> "heart")
    return lemmatizer.lemmatize(lemmatizer.lemmatize(word, "v"), "n")

def remove_stop_and_lemmatize(text):
    out = []
    for w in text.split():
        if len(w) < 2 or w in STOP:
            continue
        w = lemma(w)
        if w not in STOP:
            out.append(w)
    return " ".join(out)

df_final = pd.read_csv("data/songs_deduped.csv")   # output of step 3
df_final["lyrics_clean"] = df_final["lyrics_clean"].map(remove_stop_and_lemmatize)

empty = df_final["lyrics_clean"] == ""
print(f"Dropping {empty.sum():,} songs that were only stop words")
df_final = df_final[~empty].reset_index(drop=True)
print(f"Final dataset: {len(df_final):,} songs")

df_final.to_csv("data/songs_lyrics_cleaned.csv", index=False)
print("Saved to data/songs_lyrics_cleaned.csv")

Dropping 20 songs that were only stop words
Final dataset: 435,753 songs
Saved to data/songs_lyrics_cleaned.csv


## TF-IDF features 

### Train / test split (by artist)

In [11]:
from sklearn.model_selection import StratifiedGroupKFold

df_final = pd.read_csv("data/songs_lyrics_cleaned.csv")
df_final["primary_artist"] = df_final["artists"].map(primary_artist)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(sgkf.split(df_final, df_final["genre"], groups=df_final["primary_artist"]))
df_train = df_final.iloc[train_idx].reset_index(drop=True)
df_test = df_final.iloc[test_idx].reset_index(drop=True)

print(f"train: {len(df_train):,} songs | test: {len(df_test):,} songs")
print("artists shared between train and test:", len(set(df_train["primary_artist"]) & set(df_test["primary_artist"])))
print("\ngenre share (train vs test):")
print(pd.concat([df_train["genre"].value_counts(normalize=True).rename("train"),
                 df_test["genre"].value_counts(normalize=True).rename("test")], axis=1).round(3))

train: 348,602 songs | test: 87,151 songs
artists shared between train and test: 0

genre share (train vs test):
            train   test
genre                   
Rock        0.364  0.364
Pop         0.127  0.127
Electronic  0.124  0.124
Folk        0.094  0.094
Country     0.083  0.083
Hip-Hop     0.080  0.080
R&B         0.049  0.049
Jazz        0.029  0.029
Blues       0.027  0.027
Classical   0.023  0.023


### Unigram Tokenization

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    ngram_range=(1, 1),     
    min_df=5,               # ignore terms in fewer than 5 songs
    max_df=0.5,             # ignore terms in more than half of all songs
    max_features=300_000,   # keep the 300k most frequent terms
    sublinear_tf=True,      # 1 + log(tf): a chorus repeated 20 times shouldn't count 20x
    lowercase=False,        
    token_pattern=r"\S+",   
    dtype=np.float32,       
)

X_train = tfidf.fit_transform(df_train["lyrics_clean"])   
X_test = tfidf.transform(df_test["lyrics_clean"])        

y_train = df_train["genre"].to_numpy()
y_test = df_test["genre"].to_numpy()

print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print(f"non-zeros per song: {X_train.nnz / X_train.shape[0]:.0f}  |  sparsity: {1 - X_train.nnz / np.prod(X_train.shape):.4%}")

X_train: (348602, 47057) | X_test: (87151, 47057)
non-zeros per song: 60  |  sparsity: 99.8719%


### Save

In [13]:
import joblib
import scipy.sparse as sp

sp.save_npz("data/X_train_tfidf.npz", X_train)
sp.save_npz("data/X_test_tfidf.npz", X_test)
df_train[["name", "artists", "primary_artist", "genre"]].to_csv("data/train_labels.csv", index=False)
df_test[["name", "artists", "primary_artist", "genre"]].to_csv("data/test_labels.csv", index=False)
joblib.dump(tfidf, "model/tfidf_vectorizer.joblib")
print("Saved features and labels to data/, vectorizer to model/")

Saved features and labels to data/, vectorizer to model/


### Model Implementations

In [18]:
from sklearn.linear_model import LogisticRegression
import joblib
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV

X_train = sp.load_npz("data/X_train_tfidf.npz")
X_test = sp.load_npz("data/X_test_tfidf.npz")
train_labels = pd.read_csv("data/train_labels.csv")
test_labels = pd.read_csv("data/test_labels.csv")
y_train = train_labels["genre"].to_numpy()
y_test = test_labels["genre"].to_numpy()
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
def evaluate(name, y_true, y_pred):
    print(f"{name}: accuracy {accuracy_score(y_true, y_pred):.3f} | macro-F1 {f1_score(y_true, y_pred, average='macro'):.3f}")
    print(classification_report(y_true, y_pred, digits=3))


X_train: (348602, 47057) | X_test: (87151, 47057)


### Naive Bayes
A simple naive bayes serves as the baseline model


In [19]:
from sklearn.naive_bayes import ComplementNB
best_nb = ComplementNB().fit(X_train, y_train)
evaluate("Complement Naive Bayes", y_test, best_nb.predict(X_test))
joblib.dump(best_nb, "model/complement_nb.joblib")

Complement Naive Bayes: accuracy 0.474 | macro-F1 0.337
              precision    recall  f1-score   support

       Blues      0.360     0.066     0.112      2374
   Classical      0.444     0.073     0.125      2002
     Country      0.366     0.527     0.432      7229
  Electronic      0.423     0.194     0.266     10837
        Folk      0.608     0.274     0.378      8185
     Hip-Hop      0.410     0.858     0.555      6973
        Jazz      0.496     0.247     0.330      2557
         Pop      0.254     0.245     0.250     11051
         R&B      0.389     0.221     0.281      4232
        Rock      0.587     0.712     0.644     31711

    accuracy                          0.474     87151
   macro avg      0.434     0.342     0.337     87151
weighted avg      0.472     0.474     0.444     87151



['model/complement_nb.joblib']

### Logistic Regression

The Logistic regression is tuned using 5 fold stratified cv. It is stratified by genre and grouped by artist.

In [20]:
TUNE_N = 100_000
rng = np.random.default_rng(42)
tune_idx = rng.choice(X_train.shape[0], size=min(TUNE_N, X_train.shape[0]), replace=False)
X_tune, y_tune = X_train[tune_idx], y_train[tune_idx]
groups_tune = train_labels["primary_artist"].to_numpy()[tune_idx]

# 5-fold stratified CV (genre proportions kept in every fold), with each artist kept in a single fold
cv_splits = list(StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42).split(X_tune, y_tune, groups_tune))
for k, (tr, va) in enumerate(cv_splits, 1):
    print(f"fold {k}: train {len(tr):,} | val {len(va):,}")

lr_search = GridSearchCV(
    LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=300),
    param_grid={"C": [1, 4, 16, 64]},          # TF-IDF rows are L2-normalized, so useful C values run high
    scoring="f1_macro", cv=cv_splits, refit=False, n_jobs=-1, verbose=1,
)
lr_search.fit(X_tune, y_tune)
print(pd.DataFrame(lr_search.cv_results_)[["param_C", "mean_test_score", "std_test_score"]])
print("best:", lr_search.best_params_)

best_lr = LogisticRegression(class_weight="balanced", solver="lbfgs", max_iter=300, **lr_search.best_params_)
best_lr.fit(X_train, y_train)
evaluate("Logistic regression (tuned)", y_test, best_lr.predict(X_test))
joblib.dump(best_lr, "model/logistic_regression.joblib")

fold 1: train 79,999 | val 20,001
fold 2: train 80,000 | val 20,000
fold 3: train 80,001 | val 19,999
fold 4: train 80,000 | val 20,000
fold 5: train 80,000 | val 20,000
Fitting 5 folds for each of 4 candidates, totalling 20 fits


/Users/tylerh/Downloads/Spotify-Genre-Prediction/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/tylerh/Downloads/Spotify-Genre-Prediction/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown

   param_C  mean_test_score  std_test_score
0        1         0.397007        0.005345
1        4         0.390145        0.005880
2       16         0.376523        0.005029
3       64         0.362726        0.005170
best: {'C': 1}


/Users/tylerh/Downloads/Spotify-Genre-Prediction/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 300 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=300).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic regression (tuned): accuracy 0.475 | macro-F1 0.424
              precision    recall  f1-score   support

       Blues      0.208     0.510     0.295      2374
   Classical      0.147     0.419     0.218      2002
     Country      0.467     0.529     0.496      7229
  Electronic      0.401     0.488     0.440     10837
        Folk      0.402     0.389     0.396      8185
     Hip-Hop      0.758     0.792     0.775      6973
        Jazz      0.274     0.567     0.369      2557
         Pop      0.306     0.201     0.243     11051
         R&B      0.336     0.505     0.404      4232
        Rock      0.779     0.495     0.605     31711

    accuracy                          0.475     87151
   macro avg      0.408     0.490     0.424     87151
weighted avg      0.543     0.475     0.490     87151



['model/logistic_regression.joblib']

## Testing

In [21]:
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.preprocessing import label_binarize

classes = np.array(best_lr.classes_)
assert list(best_nb.classes_) == list(classes)

def test_metrics(name, y_true, proba):
    """Accuracy, ROC-AUC, sensitivity and specificity, one-vs-rest per genre plus macro averages."""
    y_pred = classes[proba.argmax(axis=1)]
    y_bin = label_binarize(y_true, classes=classes)
    cm = confusion_matrix(y_true, y_pred, labels=classes)

    tp = np.diag(cm)
    fn = cm.sum(axis=1) - tp
    fp = cm.sum(axis=0) - tp
    tn = cm.sum() - tp - fn - fp

    per_class = pd.DataFrame({
        "sensitivity": tp / (tp + fn),            
        "specificity": tn / (tn + fp),            
        "roc_auc": roc_auc_score(y_bin, proba, average=None),
        "support": cm.sum(axis=1),
    }, index=classes).round(3)

    summary = {
        "model": name,
        "accuracy": (y_pred == y_true).mean(),
        "roc_auc (macro OvR)": roc_auc_score(y_bin, proba, average="macro"),
        "sensitivity (macro)": per_class["sensitivity"].mean(),
        "specificity (macro)": per_class["specificity"].mean(),
    }
    print(f"\n{name}: per-genre test metrics")
    print(per_class)
    return summary

probas = {
    "Logistic regression": best_lr.predict_proba(X_test),
    "Complement NB": best_nb.predict_proba(X_test)
}
summary_df = pd.DataFrame([test_metrics(name, y_test, p) for name, p in probas.items()]).set_index("model").round(3)
print("\n=== Test set summary ===")
print(summary_df.to_string())


Logistic regression: per-genre test metrics
            sensitivity  specificity  roc_auc  support
Blues             0.510        0.946    0.869     2374
Classical         0.419        0.943    0.825     2002
Country           0.529        0.945    0.876     7229
Electronic        0.488        0.897    0.802    10837
Folk              0.389        0.940    0.792     8185
Hip-Hop           0.792        0.978    0.962     6973
Jazz              0.567        0.954    0.885     2557
Pop               0.201        0.934    0.718    11051
R&B               0.505        0.949    0.873     4232
Rock              0.495        0.920    0.828    31711

Complement NB: per-genre test metrics
            sensitivity  specificity  roc_auc  support
Blues             0.066        0.997    0.853     2374
Classical         0.073        0.998    0.770     2002
Country           0.527        0.917    0.855     7229
Electronic        0.194        0.962    0.739    10837
Folk              0.274        0.982